# FL-ATTACK-003.6 — Risk matrix and KPI verification

*Epic EPIC-001 · story fl_attack_003 · all results below are loaded from files produced by the task's experiment function and labelled MEASURED/SIMULATED; clients are virtual; data is synthetic.*

## 1. Research objective
Summarise attack/defence outcomes as a risk matrix and verify the story KPIs honestly.

## 2. Scientific background
Risk = likelihood × impact is qualitative; here scores derive from measured outcomes.

References use keys of `docs/references/references_master.md` (status per entry).

## 3. Research questions
- **RQ1.** Which KPIs are met?

## 4. Hypotheses
- **H1.** Not all KPIs are met.

## 5. Mathematical formulation
KPI check: measured vs threshold.

## 6. Experimental environment

In [1]:
import os, sys, json, platform, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
os.environ.setdefault("MLFLOW_DISABLE_AGENT_HINT", "1")
def _find_root():
    p = Path(os.environ.get("FL_ROOT") or os.environ.get("RS_ROOT") or Path.cwd()).resolve()
    for q in [p, *p.parents]:
        if (q / "src" / "federated_learning").exists():
            return q
    raise RuntimeError("repository root not found; set RS_ROOT")
ROOT = _find_root()
sys.path.insert(0, str(ROOT / "src"))
STORY = "fl_attack_003"
_cands = [ROOT / "experiments" / "results" / STORY]
RES = next(c for c in _cands if c.exists())
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, display
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
SEED = 0; np.random.seed(SEED)
def show_csv(name, n=12):
    p = RES / name
    if p.exists(): display(pd.read_csv(p).round(4).head(n))
    else: print("MISSING", name)
def show_fig(name):
    for base in (RES / "figures", RES):
        if (base / name).exists(): display(Image(filename=str(base / name))); return
    print("MISSING figure", name)
print("root:", ROOT, "| results:", RES)

root: /home/claude/fl_sprint | results: /home/claude/fl_sprint/experiments/results/fl_attack_003


In [2]:
import importlib.metadata as md
vers = {}
for p in ["numpy","scipy","scikit-learn","torch","tenseal","phe","pandas","matplotlib","cryptography","psutil","mlflow","opacus"]:
    try: vers[p] = md.version(p)
    except Exception: vers[p] = "not installed"
print(json.dumps(dict(python=platform.python_version(), platform=platform.platform(), versions=vers, seed=SEED, gpu="none (CPU only; GPU metrics NOT_MEASURED)"), indent=1))

{
 "python": "3.13.18",
 "platform": "Linux-6.18.44-fc-v338-x86_64-with-glibc2.39",
 "versions": {
  "numpy": "2.5.3",
  "scipy": "1.18.1",
  "scikit-learn": "1.9.1",
  "torch": "2.15.0",
  "tenseal": "0.3.18",
  "phe": "1.5.0",
  "pandas": "3.0.5",
  "matplotlib": "3.11.2",
  "cryptography": "50.0.1",
  "psutil": "7.2.2",
  "mlflow": "3.18.0",
  "opacus": "1.6.0"
 },
 "seed": 0,
 "gpu": "none (CPU only; GPU metrics NOT_MEASURED)"
}


## 7. Dataset description
Synthetic binary tabular dataset **SYN-IDS-v1** (scikit-learn `make_classification`, 40 features, 70:30 split). Chosen because no real IDS/clinical dataset was available in the sandbox. It has no clinical meaning; conclusions about absolute accuracy do not transfer.

In [3]:
from federated_learning.synthetic_data.generators.syn_ids_v1 import make_dataset, DATASET_ID
from federated_learning.data_partitioning.iid import partition_iid
from federated_learning.data_partitioning.dirichlet import partition_dirichlet
from federated_learning.data_partitioning.metrics import label_skew_stat
data = make_dataset(n_samples=6000, seed=SEED)
print(DATASET_ID, "train", data.X_train.shape, "test", data.X_test.shape, "positive rate", float(data.y_train.mean()))
print("NOTE: SYNTHETIC tabular data (not CIC-IDS-2017, not patient data); clients are VIRTUAL, not hospitals.")

SYN-IDS-v1 train (4200, 40) test (1800, 40) positive rate 0.40095238095238095
NOTE: SYNTHETIC tabular data (not CIC-IDS-2017, not patient data); clients are VIRTUAL, not hospitals.


## 8. Configuration and reproducibility settings
Seeds are fixed; each stored result carries `provenance` (experiment id, SHA-256 of the configuration, environment versions; `git_commit` is `null` because the workspace is not a git repository).

In [4]:
from federated_learning.training.config import FLConfig
cfg = FLConfig(rounds=5, local_epochs=1, seed=SEED)
print(cfg)

FLConfig(algo='fedavg', rounds=5, local_epochs=1, lr=0.05, batch_size=32, participation=1.0, mu=0.0, hidden=32, act='relu', seed=0, hetero_epochs=False, eval_every=1, server_lr=1.0)


## 9. Data preparation
Partitioning into virtual clients (IID or Dirichlet label skew) is performed by `federated_learning.data_partitioning`.

In [5]:
parts_iid = partition_iid(data.y_train, 10, SEED)
parts_dir = partition_dirichlet(data.y_train, 10, 0.3, SEED)
print('label-skew statistic IID:', round(label_skew_stat(data.y_train, parts_iid),3), ' Dirichlet(0.3):', round(label_skew_stat(data.y_train, parts_dir),3))

label-skew statistic IID: 0.019  Dirichlet(0.3): 0.363


## 10. Implementation
Reusable code lives in `src/federated_learning/` (`attacks.py`, `defenses.py`, `fl.py`). The cell below exercises it live (small, fast, deterministic) and asserts correctness.

In [6]:
print('loads stored KPI table')

loads stored KPI table


## 11. Experiment execution
The full experiment was executed by `python -m federated_learning.pipelines.experiments.run_all --task FL-ATTACK-003.6` (see `logs/`). Its raw per-run results are loaded here; they are not re-computed inside the notebook.

In [7]:
show_csv('task_3_6_risk_matrix.csv')

,attack,attacker,objective,assumptions,component,observed_impact,detection,mitigation,residual_risk,evidence
0,Backdoor (data poisoning + boosting),compromised hospital client(s) (20-30%),trigger-activated misclassification of attacks...,attacker controls local data & update; persist...,global model / aggregation,ASR 1.00 at 'high' vs 0.05 for clean FL (trigg...,norm/cosine anomaly filter; SMPC norm bound,best measured: anomaly_filter -> ASR 0.05,HIGH,"task_3_1_backdoor.json, task_3_5_countermeasur..."
1,Byzantine update manipulation,compromised/colluding clients (10-30%),degrade or steer the global model,attacker sends arbitrary vectors; scale 5,aggregation rule / availability,sign-flip 30%: mean aggregator acc 0.47,"median/trimmed mean/Krum, anomaly filter",best at 30% sign-flip: multi_krum acc 0.78,MEDIUM (rules rely on honest majority; not val...,task_3_2_byzantine.json
2,Membership inference (shadow-model),black-box observer / honest-but-curious server...,infer a patient record's training membership,attacker has same-distribution shadow data; ov...,global model,AUC 0.577 undefended (generalisation gap 0.12),none in protocol,lowest AUC 0.521 with dp_sigma=1.0 (utility 0....,MEDIUM,task_3_3_membership_inference.json
3,Gradient leakage (DLG / analytic),honest-but-curious server seeing individual pl...,reconstruct clinical feature vectors,"small batches, known labels, smooth activation...",client->server channel,B=1 success 1.00; successful (>10%) in 10 of 1...,none,secure aggregation / HE / SMPC hide individual...,LOW vs server when secagg/HE/SMPC used; HIGH f...,task_3_4_gradient_leakage.json


## 12. Metrics collection
Metric definitions: MEASURED over seeds: attack metric (ASR / AUC / success rate), clean accuracy, defense cost. Intensity levels low/medium/high are defined in docs/fl_attack_003/attack_methodology.md.

In [8]:
show_csv('task_3_6_kpi.csv')

,kpi,measured,met,note
0,Backdoor success rate < 5%,"0.999 (high, undefended); best defended 0.054",False,clean-FL ASR of the trigger = 0.046
1,Gradient leakage success < 10%,B=1 undefended 1.00; secagg K=10 (B_eff=20) 0.00,False,NOT met for plaintext small-batch updates; met...
2,Membership inference AUC < 0.60,undefended 0.577; lowest 0.521,True,evaluated in an overfit regime on synthetic data


## 13. Statistical analysis
95% CIs over seeds; AUC/ASR CIs may extend beyond [0,1] because they are t-intervals on few seeds (reported as computed).

In [9]:
show_csv('task_3_6_kpi.csv', 30)

,kpi,measured,met,note
0,Backdoor success rate < 5%,"0.999 (high, undefended); best defended 0.054",False,clean-FL ASR of the trigger = 0.046
1,Gradient leakage success < 10%,B=1 undefended 1.00; secagg K=10 (B_eff=20) 0.00,False,NOT met for plaintext small-batch updates; met...
2,Membership inference AUC < 0.60,undefended 0.577; lowest 0.521,True,evaluated in an overfit regime on synthetic data


## 14. Visualizations

In [10]:
print('no figure for this task (tabular result)')

no figure for this task (tabular result)


## 15. Interpretation of results
The numbers are computed in the next cell from the stored data; the text interprets only what that cell prints.

Backdoor and gradient-leakage KPIs are NOT met undefended; MIA AUC KPI met in this regime.

In [11]:
show_csv('task_3_6_kpi.csv')

,kpi,measured,met,note
0,Backdoor success rate < 5%,"0.999 (high, undefended); best defended 0.054",False,clean-FL ASR of the trigger = 0.046
1,Gradient leakage success < 10%,B=1 undefended 1.00; secagg K=10 (B_eff=20) 0.00,False,NOT met for plaintext small-batch updates; met...
2,Membership inference AUC < 0.60,undefended 0.577; lowest 0.521,True,evaluated in an overfit regime on synthetic data


## 16. Limitations
- Qualitative risk scoring.
- Synthetic data, virtual clients on one machine, 2 CPU cores, no GPU; no real network. Comm. bytes are logical payload sizes.
- No regulatory/compliance conclusion can be drawn from this notebook.

## 17. Conclusions
Honest verdicts are in the KPI table.

**Jira acceptance:** see `docs/fl_attack_003/results_and_discussion.md` for the criterion-by-criterion verdict (MET / PARTIALLY MET / NOT MET).

## 18. References
`kairouz2021` — details in `docs/fl_attack_003/references.md`.